# scarlatti-doodle project!

## Zip 파일 가상 공간에 마운트
#### this code block was written using AI

In [31]:
import io
import os
import zipfile
from music21 import converter, midi, environment

# music21 경로 억까 방지 환경 설정
try:
    environment.Environment()['directoryScratch'] = '/tmp'
except:
    pass

def mount_scarlatti():
    """scarlatti.zip을 가상 메모리에 마운트하고, 
    파일을 언제든 꺼내 쓸 수 있는 가상 폴더 객체(ZipFile)와 파일 명단을 반환합니다."""
    ZIP_FILE_PATH = "original_midi.zip"
    
    if not os.path.exists(ZIP_FILE_PATH):
        raise FileNotFoundError(f"⚠️ '{ZIP_FILE_PATH}' 파일이 없습니다. 왼쪽 탐색기에 업로드해 주세요!")
        
    print("🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...")
    
    # 파일을 RAM으로 읽어오기
    with open(ZIP_FILE_PATH, "rb") as f:
        zip_buffer = io.BytesIO(f.read())
        
    # Zip 아카이브 객체 생성
    archive = zipfile.ZipFile(zip_buffer)
    
    # 미디 파일 목록만 싹 긁어오기
    midi_files = [f for f in archive.namelist() if f.lower().endswith(('.mid', '.midi'))]
    print(f"📦 마운트 완료! 총 {len(midi_files)}개의 가상 미디 폴더 준비 완료.")
    
    return archive, midi_files

def load_midi_from_virtual_folder(archive, file_path):
    """가상 폴더(archive)와 파일 경로를 주면, 디스크 IO 없이 music21 Score 객체로 즉시 변환합니다."""
    midi_raw_bytes = archive.read(file_path)
    
    # 샌드박스 바이트 스트림 파싱 (우리가 방금 성공한 그 정석 코드)
    mf = midi.MidiFile()
    mf.readstr(midi_raw_bytes)
    return midi.translate.midiFileToStream(mf)

# 🔥 [실행] 가상 폴더 연결하기
# 앞으로 코드 짤 때는 이 두 변수(scarlatti_folder, file_list)만 가지고 노시면 됩니다!
scarlatti_folder, file_list = mount_scarlatti()
print("앞으로 scarlatti_folder, file_list에 접근해서 사용! load_midi_from_virtual_folder 함수 사용")

# 미디 시각화하는 함수 작성
from music21 import converter
import matplotlib.pyplot as plt
import numpy as np # 숫자 변환을 도와주는 도구

def ShowMidi(score):
    # 1. 데이터를 뽑을 때 아예 실수(float)로 강제 변환합니다.
    raw_pitches = [float(n.pitch.ps) for n in score.flatten().notes if getattr(n, 'isNote', False) and not n.isChord]
    raw_offsets = [float(n.offset) for n in score.flatten().notes if getattr(n, 'isNote', False) and not n.isChord]

    # 2. 혹시나 섞여 있을지 모르는 '이상한 값'들을 넘파이(numpy)로 한 번 더 걸러냅니다.
    pitches = np.array(raw_pitches, dtype=float)
    offsets = np.array(raw_offsets, dtype=float)

    # 3. 그리기
    plt.figure(figsize=(12, 4))
    plt.scatter(offsets, pitches, s=2, c='orange')
    plt.show()

    score.write()

print("ShowMidi 함수 준비됨!")

🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...
📦 마운트 완료! 총 555개의 가상 미디 폴더 준비 완료.
앞으로 scarlatti_folder, file_list에 접근해서 사용! load_midi_from_virtual_folder 함수 사용
ShowMidi 함수 준비됨!


#### 전처리 함수들 준비

In [14]:
from music21 import *
import re
import copy
import partitura as pt
import numpy as np

def Score_TrebleBassSeparation_Partitura(midi_path: str, splitPoint: str = "C4"):
    '''NoteArray 분리 후, 저장 가능한 PerformedPart 객체로 복원하여 리턴하는 함수'''
    performance = pt.load_performance_midi(midi_path)[0]
    note_array = performance.note_array()
    
    split_pitch = 60 
    
    treble_mask = note_array['pitch'] >= split_pitch
    bass_mask = note_array['pitch'] < split_pitch
    
    treble_note_array = note_array[treble_mask]
    bass_note_array = note_array[bass_mask]
    
    # 💡 [핵심 수정] NumPy 배열을 partitura가 저장할 수 있는 PerformedPart 객체 상자에 다시 담아줍니다.
    treble_part = pt.performance.PerformedPart.from_note_array(treble_note_array)
    bass_part = pt.performance.PerformedPart.from_note_array(bass_note_array)
    
    return treble_part, bass_part



# def Score_TrebleBassSeparation(inputScore: stream.Score, splitPoint: str ="C4"):
#     '''최신 music21 버전에 맞춘 오차 없는 높은음/낮은음자리 분리 함수'''
    
#     # 1. 원본 구조를 완벽하게 복사 (템포, 마디 좌표 보존)
#     treble = copy.deepcopy(inputScore)
#     bass = copy.deepcopy(inputScore)
    
#     splitPointPitch = pitch.Pitch(splitPoint)

#     # 2. 오른손 트랙: 기준보다 낮은 음표를 '구조 왜곡 없이' 정확하게 제거
#     # 악보를 훼손하지 않기 위해 리스트로 목록을 먼저 뽑아두고 지웁니다.
#     notes_to_check_treble = list(treble.recurse().notes)
    
#     for n in notes_to_check_treble:
#         if n.isChord:
#             for singleNote in list(n.notes):
#                 if singleNote.pitch < splitPointPitch:
#                     n.remove(singleNote)
#             if len(n.notes) == 0:
#                 # n.activeSite를 쓰면 마디 구조를 깨지 않고 그 상자 안에서 안전하게 쏙 빠집니다.
#                 n.activeSite.remove(n)
#         else:
#             if n.pitch < splitPointPitch:
#                 n.activeSite.remove(n)



    # 3. 왼손 트랙: 기준보다 높거나 같은 음표를 안전하게 제거
    notes_to_check_bass = list(bass.recurse().notes)
    
    for n in notes_to_check_bass:
        if n.isChord:
            for singleNote in list(n.notes):
                if singleNote.pitch >= splitPointPitch:
                    n.remove(singleNote)
            if len(n.notes) == 0:
                n.activeSite.remove(n)
        else:
            if n.pitch >= splitPointPitch:
                n.activeSite.remove(n)

    # 4. 결과 출력
    outputScore = stream.Score()
    outputScore.append(treble)
    outputScore.append(bass)

    return outputScore

def Score_TransposeToAllKeys(inputScore):
    '''스코어 파일을 모든 조로 전조해서 리턴'''
    pass

def Score_SliceByMeasures(inputScore):
    '''스코어 파일을 특정 마디 길이만큼 나눠서 리턴'''
    pass


def Score_MaskNotes(inputScore):
    '''멜로디 데이터에서 일부 데이터들 마스킹해서 리턴'''
    pass

def ScoreToDataset(inputScore):
    '''스코어 파일을 ai 학습용 데이터셋으로 변환해서 리턴, 코드 정보와 박자정보 추가'''
    pass

    


In [15]:
# 1. 함수 실행 및 완벽히 정제된 NumPy 배열 2개 받아오기
treble_data, bass_data = Score_TrebleBassSeparation_Partitura(
    midi_path="sonatas_k-531_(c)sankey.mid", 
    splitPoint="C4"
)

# 2. 결과물 확인을 위해 각각 개별 미디 파일로 안전하게 저장하기
pt.save_performance_midi(treble_data, "partitura_treble_output.mid")
pt.save_performance_midi(bass_data, "partitura_bass_output.mid")

print("✨ 파르티투라 분리 완료! 박자가 완벽하게 고정된 미디 파일이 생성되었습니다.")

✨ 파르티투라 분리 완료! 박자가 완벽하게 고정된 미디 파일이 생성되었습니다.


In [50]:
score = converter.parse("sonatas_k-531_(c)sankey.mid")

trebble = Score_TrebleBassSeparation(score)

trebble.write('midi', fp="trebble.mid")



'trebble.mid'